# Nike Executive Dashboard

Interactive Nike dashboard for `NKE`. Run the code cell below. It fetches Yahoo Finance data through `yfinance` when available and falls back to bundled Nike demo data offline.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import datetime

# -----------------------------
# Nike Executive Dashboard
# Live data source: Yahoo Finance via yfinance when available.
# Fallback data is included so the dashboard still runs offline.
# -----------------------------
TICKER = "NKE"

fallback_df = pd.DataFrame({
    "Year": [2019, 2020, 2021, 2022, 2023, 2024, 2025],
    "Revenue": [39117, 37403, 44538, 46710, 51217, 51362, 46266],
    "Net Income": [4029, 2539, 5727, 6046, 5070, 5700, 3267],
    "Gross Profit": [17512, 16241, 19962, 21508, 22292, 22828, 20148],
    "Nike Direct": [11800, 12380, 16400, 18700, 21400, 21700, 21100],
    "Wholesale": [25700, 23160, 25500, 25500, 27900, 27900, 22900],
    "Converse": [1900, 1846, 2205, 2346, 2427, 2082, 1984],
    "Corporate": [-283, 17, 433, 164, -510, -320, 282],
    "Cash": [4466, 8348, 9479, 8574, 7441, 10889, 7224],
    "Debt": [3466, 9462, 9413, 9420, 8927, 8879, 8728],
    "Market Cap": [132000, 152000, 244000, 164000, 170000, 143000, 102000],
})

fallback_stock = pd.DataFrame({
    "Date": pd.to_datetime(["2019-05-31", "2020-05-31", "2021-05-31", "2022-05-31", "2023-05-31", "2024-05-31", "2025-05-31"]),
    "Close": [80, 98, 136, 119, 106, 93, 60],
})

DATA_SOURCE = "Offline fallback Nike data"
LAST_UPDATED = datetime.now().strftime("%Y-%m-%d %H:%M")

def _to_millions(value):
    if pd.isna(value):
        return np.nan
    return float(value) / 1_000_000

def _row_value(frame, row_names, column):
    for row_name in row_names:
        if row_name in frame.index:
            value = frame.loc[row_name, column]
            if isinstance(value, pd.Series):
                value = value.iloc[0]
            return _to_millions(value)
    return np.nan

def load_yahoo_nike_data():
    """Fetch annual Nike financials and stock prices from Yahoo Finance if yfinance is installed."""
    try:
        import yfinance as yf
    except Exception as exc:
        raise RuntimeError("Install yfinance to fetch live Yahoo Finance data: pip install yfinance") from exc

    ticker = yf.Ticker(TICKER)
    financials = ticker.financials
    balance_sheet = ticker.balance_sheet

    if financials is None or financials.empty:
        raise RuntimeError("Yahoo Finance did not return annual financial statements.")

    rows = []
    for col in sorted(financials.columns):
        year = int(pd.Timestamp(col).year)
        revenue = _row_value(financials, ["Total Revenue"], col)
        net_income = _row_value(financials, ["Net Income", "Net Income Common Stockholders"], col)
        gross_profit = _row_value(financials, ["Gross Profit"], col)

        cash = np.nan
        debt = np.nan
        if balance_sheet is not None and not balance_sheet.empty and col in balance_sheet.columns:
            cash = _row_value(balance_sheet, ["Cash And Cash Equivalents", "Cash Cash Equivalents And Short Term Investments"], col)
            debt = _row_value(balance_sheet, ["Total Debt", "Long Term Debt And Capital Lease Obligation"], col)

        rows.append({
            "Year": year,
            "Revenue": revenue,
            "Net Income": net_income,
            "Gross Profit": gross_profit,
            "Cash": cash,
            "Debt": debt,
        })

    yahoo_df = pd.DataFrame(rows).dropna(subset=["Revenue"]).sort_values("Year").tail(7)
    if yahoo_df.empty:
        raise RuntimeError("Yahoo Finance returned no usable annual rows.")

    # Yahoo Finance statements do not reliably expose Nike channel revenue, so use Nike-style segment estimates.
    segment_cols = ["Nike Direct", "Wholesale", "Converse", "Corporate"]
    yahoo_df = yahoo_df.merge(fallback_df[["Year"] + segment_cols], on="Year", how="left")
    for col in segment_cols:
        if yahoo_df[col].isna().any():
            ratio = fallback_df[col].sum() / fallback_df["Revenue"].sum()
            yahoo_df[col] = yahoo_df[col].fillna(yahoo_df["Revenue"] * ratio)

    try:
        fast_info = ticker.fast_info
        market_cap = getattr(fast_info, "market_cap", None) or fast_info.get("market_cap")
    except Exception:
        market_cap = None
    yahoo_df["Market Cap"] = np.nan
    yahoo_df.loc[yahoo_df.index[-1], "Market Cap"] = _to_millions(market_cap) if market_cap else fallback_df["Market Cap"].iloc[-1]
    yahoo_df["Market Cap"] = yahoo_df["Market Cap"].interpolate().bfill().ffill()

    stock = ticker.history(period="5y", interval="1mo", auto_adjust=True).reset_index()
    if stock is None or stock.empty:
        stock = fallback_stock.copy()
    stock["Date"] = pd.to_datetime(stock["Date"]).dt.tz_localize(None)

    return yahoo_df.reset_index(drop=True), stock[["Date", "Close"]].dropna(), "Yahoo Finance via yfinance"

try:
    df, stock_df, DATA_SOURCE = load_yahoo_nike_data()
except Exception as err:
    df = fallback_df.copy()
    stock_df = fallback_stock.copy()
    DATA_SOURCE = f"Offline fallback Nike data ({err})"

metric_cols = ["Revenue", "Net Income", "Gross Profit", "Nike Direct", "Wholesale", "Converse", "Corporate", "Cash", "Debt", "Market Cap"]
for col in metric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

def pct_change(data, column, selected_value):
    if selected_value == "All":
        first_value = data.loc[data["Year"] == data["Year"].min(), column].iloc[0]
        last_value = data.loc[data["Year"] == data["Year"].max(), column].iloc[0]
        if first_value == 0:
            return "No base value"
        change = (last_value - first_value) / first_value * 100
        return f"{change:+.1f}% overall change"

    current = data.loc[data["Year"] == selected_value, column].iloc[0]
    previous = data.loc[data["Year"] < selected_value, column]
    if previous.empty or previous.iloc[-1] == 0:
        return "Base year"
    change = (current - previous.iloc[-1]) / previous.iloc[-1] * 100
    return f"{change:+.1f}% vs previous year"

def money(value):
    return f"$ {value:,.0f} M"

def draw_dashboard(selected_year):
    if selected_year == "All":
        row = df.iloc[-1]
        data = df.copy()
        dashboard_title = "All Years Combined"
    else:
        row = df[df["Year"] == selected_year].iloc[0]
        data = df[df["Year"] <= selected_year]
        dashboard_title = selected_year

    display(HTML(f"""
    <div style="
        background:#050505;
        border:1px solid #222;
        border-radius:8px;
        padding:20px 24px;
        font-family:Arial, Helvetica, sans-serif;
        color:white;
        margin-bottom:14px;
        box-shadow:0 14px 30px rgba(0,0,0,.32);
    ">
        <div style="display:flex; align-items:center; gap:24px; flex-wrap:wrap;">
            <div style="
                background:#fff;
                color:#050505;
                padding:18px 24px;
                border-radius:8px;
                min-width:270px;
                box-shadow:0 8px 24px rgba(255,255,255,.12);
            ">
                <div style="display:flex; align-items:center; justify-content:center; gap:12px;">
                    <svg width="88" height="42" viewBox="0 0 300 140" aria-label="Nike swoosh">
                        <path fill="#050505" d="M292 28C225 56 155 90 87 102 56 108 32 103 23 88 14 73 24 51 47 28 36 45 35 58 44 65 55 74 79 71 110 60 166 41 223 20 292 28Z"/>
                    </svg>
                    <div style="font-size:36px; font-weight:900; letter-spacing:1px;">NIKE</div>
                </div>
                <div style="text-align:center; font-size:13px; color:#333; font-weight:800; margin-top:4px;">
                    Just Do It
                </div>
            </div>
            <div style="flex:1; min-width:320px;">
                <div style="font-size:32px; font-weight:900; letter-spacing:0;">Nike Executive Dashboard</div>
                <div style="font-size:15px; color:#D1D5DB; margin-top:6px;">
                    Revenue, margins, channel mix, liquidity, market value and stock movement for NKE
                </div>
                <div style="font-size:12px; color:#F97316; margin-top:8px; font-weight:800;">
                    Source: {DATA_SOURCE} | Refreshed: {LAST_UPDATED}
                </div>
            </div>
        </div>
    </div>
    """))

    if selected_year == "All":
        revenue_value = df["Revenue"].sum()
        income_value = df["Net Income"].sum()
        direct_value = df["Nike Direct"].sum()
        market_value = df["Market Cap"].iloc[-1]
    else:
        revenue_value = row["Revenue"]
        income_value = row["Net Income"]
        direct_value = row["Nike Direct"]
        market_value = row["Market Cap"]

    kpis = [
        ("Revenue", money(revenue_value), pct_change(df, "Revenue", selected_year), "#FFFFFF"),
        ("Net Income", money(income_value), pct_change(df, "Net Income", selected_year), "#F97316"),
        ("Nike Direct", money(direct_value), pct_change(df, "Nike Direct", selected_year), "#EF4444"),
        ("Market Cap", money(market_value), pct_change(df, "Market Cap", selected_year), "#FACC15"),
    ]

    kpi_html = ""
    for title, value, delta, color in kpis:
        kpi_html += f"""
        <div style="
            background:#0B0B0B;
            border:1px solid #2A2A2A;
            border-radius:8px;
            padding:16px;
            color:white;
            box-shadow:0 10px 24px rgba(0,0,0,.22);
        ">
            <div style="font-size:12px; color:#9CA3AF; font-weight:900; text-transform:uppercase;">{title}</div>
            <div style="font-size:28px; font-weight:900; margin-top:8px; color:{color};">{value}</div>
            <div style="font-size:13px; color:#D1D5DB; margin-top:6px;">{delta}</div>
        </div>
        """

    display(HTML(f"""
    <div style="display:grid; grid-template-columns:repeat(4,minmax(160px,1fr)); gap:14px; font-family:Arial; margin-bottom:16px;">
        {kpi_html}
    </div>
    """))

    plt.style.use("dark_background")
    fig = plt.figure(figsize=(18, 10), facecolor="#050505")
    gs = fig.add_gridspec(2, 3, hspace=0.38, wspace=0.28)

    chart_bg = "#0B0B0B"
    grid_color = "#303030"
    text_color = "#E5E7EB"
    white = "#FFFFFF"
    orange = "#F97316"
    red = "#EF4444"
    amber = "#FACC15"
    gray = "#9CA3AF"
    blue = "#38BDF8"
    green = "#22C55E"

    def style_ax(ax, title):
        ax.set_facecolor(chart_bg)
        ax.set_title(title, color="white", fontsize=14, fontweight="bold", pad=12)
        ax.tick_params(colors=text_color)
        ax.grid(True, color=grid_color, alpha=0.55)
        for spine in ax.spines.values():
            spine.set_color("#3A3A3A")

    ax1 = fig.add_subplot(gs[0, 0])
    style_ax(ax1, "Revenue and Net Income Trend")
    ax1.bar(data["Year"], data["Revenue"], color=white, label="Revenue")
    ax1.set_ylabel("Revenue $M", color=white)
    ax1.tick_params(axis="y", labelcolor=white)

    ax1_income = ax1.twinx()
    ax1_income.plot(data["Year"], data["Net Income"], color=orange, marker="o", linewidth=3, label="Net Income")
    ax1_income.set_ylabel("Net Income $M", color=orange)
    ax1_income.tick_params(axis="y", labelcolor=orange)
    ax1_income.set_facecolor(chart_bg)
    for spine in ax1_income.spines.values():
        spine.set_color("#3A3A3A")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1_income.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, facecolor=chart_bg, edgecolor="#3A3A3A", loc="upper left")

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_facecolor(chart_bg)
    ax2.set_title("Channel and Brand Mix" if selected_year == "All" else f"Channel and Brand Mix - {selected_year}", color="white", fontsize=14, fontweight="bold", pad=12)
    segment_cols = ["Nike Direct", "Wholesale", "Converse", "Corporate"]
    segment_values = [df[col].sum() if selected_year == "All" else row[col] for col in segment_cols]
    segment_values = [max(0, value) for value in segment_values]
    wedges, texts, autotexts = ax2.pie(
        segment_values,
        labels=["Nike Direct", "Wholesale", "Converse", "Corporate"],
        autopct="%1.1f%%",
        startangle=90,
        colors=[orange, white, red, gray],
        wedgeprops=dict(width=0.42, edgecolor=chart_bg),
    )
    for t in texts + autotexts:
        t.set_color("white")
        t.set_fontsize(9)
    ax2.text(0, 0, "NKE", ha="center", va="center", color="white", fontsize=16, fontweight="bold")

    ax3 = fig.add_subplot(gs[0, 2])
    style_ax(ax3, "Channel Revenue Trend")
    ax3.plot(data["Year"], data["Nike Direct"], color=orange, marker="o", linewidth=3, label="Nike Direct")
    ax3.plot(data["Year"], data["Wholesale"], color=white, marker="o", linewidth=3, label="Wholesale")
    ax3.plot(data["Year"], data["Converse"], color=red, marker="o", linewidth=2.5, label="Converse")
    ax3.set_ylabel("$M")
    ax3.legend(facecolor=chart_bg, edgecolor="#3A3A3A", fontsize=8)

    ax4 = fig.add_subplot(gs[1, 0])
    style_ax(ax4, "Gross Profit, Cash and Debt")
    ax4.plot(data["Year"], data["Gross Profit"], color=amber, marker="o", linewidth=3, label="Gross Profit")
    ax4.fill_between(data["Year"], data["Cash"], color=green, alpha=0.35, label="Cash")
    ax4.fill_between(data["Year"], data["Debt"], color=red, alpha=0.35, label="Debt")
    ax4.set_ylabel("$M")
    ax4.legend(facecolor=chart_bg, edgecolor="#3A3A3A", fontsize=8)

    ax5 = fig.add_subplot(gs[1, 1])
    style_ax(ax5, "Margin Analysis")
    gross_margin = data["Gross Profit"] / data["Revenue"].replace(0, np.nan) * 100
    net_margin = data["Net Income"] / data["Revenue"].replace(0, np.nan) * 100
    ax5.plot(data["Year"], gross_margin, color=orange, marker="o", linewidth=3, label="Gross Margin")
    ax5.plot(data["Year"], net_margin, color=white, marker="o", linewidth=3, label="Net Margin")
    ax5.set_ylabel("%")
    ax5.legend(facecolor=chart_bg, edgecolor="#3A3A3A")

    ax6 = fig.add_subplot(gs[1, 2])
    style_ax(ax6, "NKE Stock Price - 5Y Monthly")
    filtered_stock = stock_df.copy()
    if selected_year != "All":
        filtered_stock = filtered_stock[filtered_stock["Date"].dt.year <= selected_year]
    ax6.plot(filtered_stock["Date"], filtered_stock["Close"], color=blue, linewidth=3, label="Adjusted Close")
    ax6.fill_between(filtered_stock["Date"], filtered_stock["Close"], color=blue, alpha=0.18)
    ax6.set_ylabel("Price $")
    ax6.legend(facecolor=chart_bg, edgecolor="#3A3A3A", fontsize=8)

    fig.suptitle(f"Nike Dashboard - {dashboard_title}", fontsize=22, fontweight="bold", color="white", y=0.99)
    plt.show()

year_filter = widgets.Dropdown(
    options=["All"] + sorted(df["Year"].astype(int).tolist(), reverse=True),
    value="All",
    description="Year Filter:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)

dashboard_output = widgets.Output()

def update_dashboard(change=None):
    with dashboard_output:
        clear_output(wait=True)
        draw_dashboard(year_filter.value)

year_filter.observe(update_dashboard, names="value")

display(year_filter)
display(dashboard_output)
update_dashboard()
